# DAPO/GRPO Training for Nemotron Nano 3.5

Runnable companion to `grpo_training_cookbook.md`. Complete the public setup in the parent `README.md`, export `SHARED_ROOT`, and run this notebook on a four-GPU system.

In [ ]:
import os
from pathlib import Path

shared_value = os.environ.get('SHARED_ROOT')
if not shared_value:
    raise RuntimeError('Export SHARED_ROOT before starting Jupyter.')

SHARED_ROOT = Path(shared_value).expanduser().resolve()
NEMOTRON_REPO = Path(os.environ.get('NEMOTRON_REPO', SHARED_ROOT / 'code/Nemotron')).expanduser().resolve()
NEMO_RL_IMAGE = os.environ.get('NEMO_RL_IMAGE', 'nemo-rl:nemotron-nano-3.5')
MODEL = SHARED_ROOT / 'models/nemotron-nano-3.5-ea2'
RECIPE = NEMOTRON_REPO / 'usage-cookbook/Nemotron-Nano-3.5/RL/grpo-dapo/dapo_nano_3_5_starter.yaml'

try:
    recipe_relative = RECIPE.relative_to(SHARED_ROOT)
except ValueError as exc:
    raise RuntimeError('NEMOTRON_REPO must be located under SHARED_ROOT.') from exc

RECIPE_CONTAINER = Path('/shared') / recipe_relative
assert RECIPE.is_file(), RECIPE
assert (MODEL / 'config.json').is_file(), MODEL

os.environ['SHARED_ROOT'] = str(SHARED_ROOT)
os.environ['NEMO_RL_IMAGE'] = NEMO_RL_IMAGE
os.environ['RECIPE_CONTAINER'] = str(RECIPE_CONTAINER)
print(f'Recipe: {RECIPE}')
print(f'Model: {MODEL}')
print(f'Image: {NEMO_RL_IMAGE}')

## Verify the runtime

The supplied recipe expects four visible GPUs.

In [ ]:
%%bash
set -euo pipefail
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
docker image inspect "$NEMO_RL_IMAGE" --format '{{.Id}}'

## Run one optimizer step

This smoke test performs generation, DAPO reward verification, log-probability calculation, and a policy update.

In [ ]:
%%bash
set -euo pipefail
mkdir -p \
  "$SHARED_ROOT/.cache/huggingface" \
  "$SHARED_ROOT/logs/dapo_nano_3_5_starter" \
  "$SHARED_ROOT/results/dapo_nano_3_5_starter"

docker run --rm --gpus all --ipc=host \
  --ulimit memlock=-1 --ulimit stack=67108864 \
  -e CUDA_VISIBLE_DEVICES=0,1,2,3 \
  -e HF_HOME=/shared/.cache/huggingface \
  -e HF_TOKEN \
  -v "$SHARED_ROOT:/shared" -w /opt/nemo-rl "$NEMO_RL_IMAGE" \
  /opt/nemo_rl_venv/bin/python examples/run_grpo.py \
  --config "$RECIPE_CONTAINER" \
  grpo.max_num_steps=1 \
  grpo.num_prompts_per_step=1 \
  grpo.num_generations_per_prompt=4 \
  grpo.val_period=-1 \
  grpo.reward_shaping.max_response_length=256 \
  policy.train_global_batch_size=4 \
  policy.max_total_sequence_length=1024 \
  policy.generation.max_new_tokens=256 \
  data.max_input_seq_length=768 \
  data.validation=null \
  env.math.num_workers=2 \
  checkpointing.enabled=false \
  logger.monitor_gpus=false \
  logger.tensorboard_enabled=false

A successful run generates four responses and completes one policy update. Smoke-test rewards and losses are wiring checks, not model-quality measurements.